# Khảo sát dữ liệu KTVIC

Chỉ train/validation; không đọc ảnh hay captions official test.
Chạy prepare_data.py trước nếu chưa có split. Không commit output chứa ảnh/caption gốc.


In [ ]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
# Có thể đặt KTVIC_DATA_DIR trỏ đến dữ liệu local đã chuẩn bị.
DATA = Path(os.environ.get('KTVIC_DATA_DIR', str(ROOT / 'data')))
SPLIT = DATA / 'splits/ktvic_seed42_content_safe.json'
from scripts.check_week1 import inspect_data
report = inspect_data(DATA, SPLIT)
report


In [ ]:
import matplotlib.pyplot as plt
from src.data.annotations import read_annotations, load_train_val_split
from src.data.tokenizer import tokenize
raw = read_annotations(DATA / 'train_data.json')
split = load_train_val_split(DATA, SPLIT)
fig, ax = plt.subplots(figsize=(8, 4))
for name in ('train', 'val'):
    ids = set(split[f'{name}_image_ids'])
    lengths = [len(tokenize(a['segment_caption'])) for a in raw['annotations'] if a['image_id'] in ids]
    ax.hist(lengths, bins=25, alpha=.5, density=True, label=name)
ax.set(xlabel='Số token', ylabel='Mật độ', title='Độ dài caption theo partition')
ax.legend()
plt.show()


In [ ]:
from PIL import Image
from IPython.display import display
image_id = split['train_image_ids'][0]
filename = next(i['filename'] for i in raw['images'] if i['id'] == image_id)
with Image.open(DATA / 'train-images' / filename) as image:
    preview = image.convert('RGB')
    preview.thumbnail((640, 480))
    display(preview)
for row in raw['annotations']:
    if row['image_id'] == image_id:
        print(row['caption'])


## Nhận xét của nhóm

Bổ sung sau khi chạy: độ dài phổ biến, OOV, caption thiếu, ảnh khó và rủi ro crop/lật ảnh làm sai mô tả hướng.
Không kết luận chất lượng mô hình từ EDA hoặc batch smoke.
